# Topo-Brain Validation Pipeline

**Checkpoint: 110,000 iterations**  
**Test subjects: sub-09, sub-10 (frozen held-out)**

Run each section in order. All cells call the existing scripts — nothing is reimplemented here.

## Steps
1. [Setup & Paths](#1-setup)
2. [Step 0: Unzip Checkpoint](#2-unzip)
3. [Step 1: Freeze Test Split](#3-test-split)
4. [Step 2: Full-Volume Evaluation (SSIM/PSNR/Dice/HD95)](#4-eval)
5. [Step 3: Aggregate Metrics](#5-aggregate)
6. [Step 4: MRIQC Install & Run](#6-mriqc)
7. [Step 5: FastSurfer Install & Hippocampal Segmentation](#7-fastsurfer)
8. [Step 6: Blinded Reader Study Prep](#8-blinded)
9. [Step 7: Regional Temporal Lobe Analysis](#9-temporal)
10. [Final Results Summary](#10-summary)

## 1. Setup & Paths <a id='1-setup'></a>

**Edit these paths before running anything else.**

In [ ]:
import os
import subprocess
import json
from pathlib import Path

# ============================================================
# CONFIGURE THESE PATHS
# ============================================================

# Root of the Topo-Brain project
PROJECT_ROOT = Path(r'D:/BCT_pcampus/Semester VII/Major Project/topobrain/Topo-Brain')

# Where the checkpoint_110000.zip was downloaded
CHECKPOINT_ZIP = Path(r'C:/Users/Asus/Downloads/checkpoint_110000.zip')

# Where the checkpoint will be extracted to
CHECKPOINT_DIR = PROJECT_ROOT / 'output' / 'checkpoint_110000'

# Base directory on the SERVER where your MRI data lives
# (the path that contains sub-01/, sub-02/, etc.)
DATA_ROOT = Path('/eos/home-i04/p/ppokhrel/data/topobrain/')  # <-- UPDATE THIS

# Path to pairs_new.csv
PAIRS_CSV = PROJECT_ROOT / 'pairs_new.csv'

# Where results will be saved
RESULTS_DIR = PROJECT_ROOT / 'results'

# Test subjects (frozen held-out)
TEST_SUBJECTS = ['sub-09', 'sub-10']

# FreeSurfer license (required for FastSurfer)
FS_LICENSE = Path('~/license.txt').expanduser()  # <-- UPDATE THIS

# ============================================================

os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Checkpoint zip:', CHECKPOINT_ZIP, '| exists:', CHECKPOINT_ZIP.exists())
print('Data root:', DATA_ROOT, '| exists:', DATA_ROOT.exists())
print('Pairs CSV:', PAIRS_CSV, '| exists:', PAIRS_CSV.exists())
print('Results dir:', RESULTS_DIR)
print('Test subjects:', TEST_SUBJECTS)

---
## Step 0: Unzip Checkpoint <a id='2-unzip'></a>

In [ ]:
import zipfile

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT_ZIP.exists():
    print(f'ERROR: Checkpoint zip not found: {CHECKPOINT_ZIP}')
else:
    with zipfile.ZipFile(CHECKPOINT_ZIP, 'r') as z:
        z.extractall(CHECKPOINT_DIR)
    print(f'Extracted to: {CHECKPOINT_DIR}')
    for f in sorted(CHECKPOINT_DIR.rglob('*.pt')):
        print(f'  Found: {f}')

# Find the checkpoint .pt file
pt_files = sorted(CHECKPOINT_DIR.rglob('*.pt'))
CHECKPOINT_PT = pt_files[0] if pt_files else None
print(f'\nCheckpoint file: {CHECKPOINT_PT}')

---
## Step 1: Freeze Test Split <a id='3-test-split'></a>

The test split is already frozen in `test_split.json`. This cell just verifies it.

In [ ]:
split_file = PROJECT_ROOT / 'test_split.json'

with open(split_file) as f:
    split = json.load(f)

print('Frozen test split:')
print(f"  Train: {split['train']}")
print(f"  Test:  {split['test']}")
print(f"  Frozen date: {split['_frozen_date']}")
print(f"  Checkpoint at freeze: {split['_checkpoint_at_freeze']}")
print()
print('IMPORTANT: Do NOT use test subjects in training.')
print('These subjects will only be used for final evaluation.')

---
## Step 2: Full-Volume Evaluation (SSIM / PSNR / Dice / HD95) <a id='4-eval'></a>

Runs `evaluate_full_volume.py` on each test subject. This is the main quantitative evaluation.

**Time:** ~5-15 minutes per subject (depends on GPU)

In [ ]:
# Verify checkpoint and data root are set correctly
if CHECKPOINT_PT is None:
    print('ERROR: No checkpoint .pt file found. Run Step 0 first.')
elif not DATA_ROOT.exists():
    print(f'ERROR: DATA_ROOT not found: {DATA_ROOT}')
    print('Update DATA_ROOT in the Setup cell.')
else:
    print(f'Checkpoint: {CHECKPOINT_PT}')
    print(f'Data root:  {DATA_ROOT}')
    print(f'Pairs CSV:  {PAIRS_CSV}')
    print('\nReady to evaluate.')

In [ ]:
# Run evaluation for each test subject
for subject in TEST_SUBJECTS:
    out_dir = RESULTS_DIR / subject
    print(f'\n{"="*60}')
    print(f'Evaluating {subject} ...')
    print(f'Output: {out_dir}')
    print(f'{"="*60}')
    
    cmd = [
        'python', 'scripts/evaluate_full_volume.py',
        '--checkpoint', str(CHECKPOINT_PT),
        '--subject', subject,
        '--pairs_csv', str(PAIRS_CSV),
        '--data-root', str(DATA_ROOT),
        '--output_dir', str(out_dir),
    ]
    
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True)
    
    if result.returncode == 0:
        print(f'\n  SUCCESS: {subject} evaluation complete')
        metrics_file = out_dir / 'metrics.json'
        if metrics_file.exists():
            with open(metrics_file) as f:
                m = json.load(f)
            print(f'  SSIM: {m.get("ssim", "N/A"):.4f}')
            print(f'  PSNR: {m.get("psnr", "N/A"):.2f} dB')
            print(f'  Dice: {m.get("dice", "N/A"):.4f}')
            print(f'  HD95: {m.get("hd95_mm", "N/A"):.2f} mm')
    else:
        print(f'  ERROR: Evaluation failed for {subject}. Check output above.')

---
## Step 3: Aggregate Metrics <a id='5-aggregate'></a>

In [ ]:
result = subprocess.run(
    ['python', 'scripts/aggregate_metrics.py',
     '--results-dir', str(RESULTS_DIR),
     '--output', str(RESULTS_DIR / 'test_metrics.csv')],
    cwd=PROJECT_ROOT, text=True
)

# Also display the CSV
csv_path = RESULTS_DIR / 'test_metrics.csv'
if csv_path.exists():
    import csv
    with open(csv_path) as f:
        print(f.read())

---
## Step 4: MRIQC — Image Quality Metrics <a id='6-mriqc'></a>

MRIQC computes IQMs (CJV, CNR, EFC, FBER, SNR) on 3T, synthetic 7T, and real 7T.

### Install MRIQC (run once)

In [ ]:
# Check if mriqc is installed
result = subprocess.run(['mriqc', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'MRIQC is installed: {result.stdout.strip()}')
else:
    print('MRIQC not installed.')
    print('\nInstall with:')
    print('  pip install mriqc')
    print('\nOr using Docker (no installation needed):')
    print('  docker pull nipreps/mriqc:latest')
    print('  Then add --docker flag to the run_mriqc.py calls below')

In [ ]:
# Install MRIQC if needed (uncomment and run)
# subprocess.run(['pip', 'install', 'mriqc'], check=True)

In [ ]:
# Run MRIQC on 3T input
subprocess.run(
    ['python', 'scripts/run_mriqc.py',
     '--mode', '3t',
     '--subjects'] + TEST_SUBJECTS + [
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / '3t'),
     # '--docker',  # Uncomment if using Docker
    ],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run MRIQC on synthetic 7T
subprocess.run(
    ['python', 'scripts/run_mriqc.py',
     '--mode', 'synthetic7t',
     '--subjects'] + TEST_SUBJECTS + [
     '--results-dir', str(RESULTS_DIR),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / 'synthetic7t'),
    ],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run MRIQC on real 7T
subprocess.run(
    ['python', 'scripts/run_mriqc.py',
     '--mode', 'real7t',
     '--subjects'] + TEST_SUBJECTS + [
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / 'real7t'),
    ],
    cwd=PROJECT_ROOT
)

In [ ]:
# Compare IQMs across all three conditions
subprocess.run(
    ['python', 'scripts/run_mriqc.py',
     '--mode', 'compare',
     '--mriqc-dir', str(RESULTS_DIR / 'mriqc'),
    ],
    cwd=PROJECT_ROOT
)

---
## Step 5: FastSurfer — Hippocampal / MTL Segmentation <a id='7-fastsurfer'></a>

FastSurfer extracts hippocampal volumes and entorhinal cortex morphometry.

### Install FastSurfer (run once on the server)

```bash
# Option A: Docker (easiest)
docker pull deepmi/fastsurfer:latest

# Option B: Singularity (for HPC/CERN)
singularity pull fastsurfer.sif docker://deepmi/fastsurfer:latest

# FreeSurfer license (required, free):
# Register at: https://surfer.nmr.mgh.harvard.edu/registration.html
# Download license.txt and copy to ~/license.txt
```

In [ ]:
FASTSURFER_DIR = RESULTS_DIR / 'fastsurfer'

# Check FreeSurfer license
if not FS_LICENSE.exists():
    print(f'WARNING: FreeSurfer license not found: {FS_LICENSE}')
    print('Download from: https://surfer.nmr.mgh.harvard.edu/registration.html')
    print(f'Save to: {FS_LICENSE}')
else:
    print(f'FreeSurfer license found: {FS_LICENSE}')

In [ ]:
# Run FastSurfer on synthetic 7T (model output)
subprocess.run(
    ['python', 'scripts/run_fastsurfer.py',
     '--mode', 'synthetic7t',
     '--subjects'] + TEST_SUBJECTS + [
     '--results-dir', str(RESULTS_DIR),
     '--fastsurfer-dir', str(FASTSURFER_DIR),
     '--fs-license', str(FS_LICENSE),
     '--docker',      # remove if using singularity or local install
     '--seg-only',    # faster: skips surface reconstruction
    ],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run FastSurfer on real 7T (ground truth)
subprocess.run(
    ['python', 'scripts/run_fastsurfer.py',
     '--mode', 'real7t',
     '--subjects'] + TEST_SUBJECTS + [
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--fastsurfer-dir', str(FASTSURFER_DIR),
     '--fs-license', str(FS_LICENSE),
     '--docker',
     '--seg-only',
    ],
    cwd=PROJECT_ROOT
)

In [ ]:
# Compare hippocampal volumes: synthetic 7T vs real 7T
subprocess.run(
    ['python', 'scripts/run_fastsurfer.py',
     '--mode', 'compare',
     '--subjects'] + TEST_SUBJECTS + [
     '--fastsurfer-dir', str(FASTSURFER_DIR),
    ],
    cwd=PROJECT_ROOT
)

---
## Step 6: Blinded Reader Study Prep <a id='8-blinded'></a>

Generates randomized comparison panels so a reader can score synthetic vs real 7T without knowing which is which.

In [ ]:
subprocess.run(
    ['python', 'scripts/blinded_reader_prep.py',
     '--subjects'] + TEST_SUBJECTS + [
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--results-dir', str(RESULTS_DIR),
     '--output-dir', str(RESULTS_DIR / 'blinded_study'),
    ],
    cwd=PROJECT_ROOT
)

print('\nBlinded panels generated.')
print(f'Share panels/single/ with the reader.')
print(f'Reader fills in: results/blinded_study/scoresheet.csv')

---
## Step 7: Regional Temporal Lobe Analysis <a id='9-temporal'></a>

After FastSurfer runs, extract entorhinal cortex + parahippocampal gyrus metrics.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

# Load morphometry comparison JSON
morph_file = RESULTS_DIR / 'fastsurfer' / 'morphometry_comparison.json'

if not morph_file.exists():
    print(f'Run FastSurfer steps first. Expected file: {morph_file}')
else:
    with open(morph_file) as f:
        morph = json.load(f)

    print('=' * 70)
    print('TEMPORAL LOBE REGIONAL ANALYSIS')
    print('=' * 70)

    temporal_regions = [
        'lh_entorhinal_vol_mm3', 'rh_entorhinal_vol_mm3',
        'lh_parahippocampal_vol_mm3', 'rh_parahippocampal_vol_mm3',
        'lh_entorhinal_thick_mm', 'rh_entorhinal_thick_mm',
        'lh_fusiform_vol_mm3', 'rh_fusiform_vol_mm3',
    ]

    import numpy as np

    for metric in temporal_regions:
        synth_vals = [morph.get('synthetic7t', {}).get(s, {}).get(metric) for s in TEST_SUBJECTS]
        real_vals  = [morph.get('real7t', {}).get(s, {}).get(metric)  for s in TEST_SUBJECTS]

        synth_vals = [v for v in synth_vals if v is not None]
        real_vals  = [v for v in real_vals  if v is not None]

        if not synth_vals or not real_vals:
            continue

        s_mean = np.mean(synth_vals)
        r_mean = np.mean(real_vals)
        diff_pct = 100 * (s_mean - r_mean) / r_mean if r_mean != 0 else float('nan')

        flag = ''
        if abs(diff_pct) > 15:
            flag = ' <-- LARGE DIFFERENCE'
        elif abs(diff_pct) > 8:
            flag = ' <-- moderate difference'

        label = metric.replace('_', ' ').title()
        print(f'{label:<40} synth={s_mean:>8.1f}  real={r_mean:>8.1f}  diff={diff_pct:>+6.1f}%{flag}')

    print()
    print('Interpretation:')
    print('  < 8%  diff: Excellent temporal lobe fidelity')
    print('  8-15% diff: Acceptable, but note limitations')
    print('  > 15% diff: Poor - consider increasing lambda_topo or training longer')

---
## Step 10: Final Results Summary <a id='10-summary'></a>

In [ ]:
print('=' * 70)
print('TOPO-BRAIN VALIDATION SUMMARY — CHECKPOINT 110k')
print('=' * 70)

# --- Quantitative metrics ---
csv_path = RESULTS_DIR / 'test_metrics.csv'
print('\n1. QUANTITATIVE METRICS (SSIM/PSNR/Dice/HD95)')
if csv_path.exists():
    import csv
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    for row in rows:
        print(f"   {row['subject']}: SSIM={row['ssim']}, PSNR={row['psnr']}, "
              f"Dice={row['dice']}, HD95={row['hd95_mm']}")
else:
    print('   Not yet computed. Run Step 2 & 3.')

# --- MRIQC ---
print('\n2. MRIQC IQMs')
iqm_csv = RESULTS_DIR / 'mriqc' / 'iqm_comparison.csv'
if iqm_csv.exists():
    with open(iqm_csv) as f:
        print(f.read()[:1000])
else:
    print('   Not yet computed. Run Step 4.')

# --- Hippocampal volumes ---
print('\n3. HIPPOCAMPAL VOLUMES')
morph_file = RESULTS_DIR / 'fastsurfer' / 'morphometry_comparison.json'
if morph_file.exists():
    with open(morph_file) as f:
        morph = json.load(f)
    for subject in TEST_SUBJECTS:
        synth = morph.get('synthetic7t', {}).get(subject, {})
        real = morph.get('real7t', {}).get(subject, {})
        s_hipp = synth.get('hippocampus_total_mm3', 'N/A')
        r_hipp = real.get('hippocampus_total_mm3', 'N/A')
        print(f'   {subject}: Synthetic={s_hipp}  Real={r_hipp}')
else:
    print('   Not yet computed. Run Step 5.')

# --- Blinded reader ---
print('\n4. BLINDED READER STUDY')
scoresheet = RESULTS_DIR / 'blinded_study' / 'scoresheet.csv'
if scoresheet.exists():
    print(f'   Panels ready at: results/blinded_study/panels/single/')
    print(f'   Reader scoresheet: {scoresheet}')
    print('   Status: Waiting for reader scores...')
else:
    print('   Not yet prepared. Run Step 6.')

print()
print('=' * 70)
print('All steps complete? You are ready for the AD classifier phase.')
print('=' * 70)